# Feature Engineering for Bank Marketing Data

## 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import requests # Added for downloading
import matplotlib.pyplot as plt # For plotting
import seaborn as sns # For enhanced visualizations
from sklearn.preprocessing import StandardScaler # For scaling numerical features
from sklearn.feature_selection import mutual_info_classif, SelectKBest # For Feature Selection
import json # For saving selected features list

# Display plots inline
%matplotlib inline

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Download and Load the Data

In [ ]:
# Dataset URL
url = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
zip_file_name = 'bank_marketing.zip'
csv_file_name_in_zip = 'bank-full.csv' 
desired_csv_output_path = 'bank-full.csv'
data_dir = './'

if not os.path.exists(desired_csv_output_path):
    print(f"Starting download from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    with open(zip_file_name, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded '{zip_file_name}' successfully.")

    extracted_correctly = False
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        print(f"\nExtracting '{csv_file_name_in_zip}' from '{zip_file_name}'...")
        file_to_extract = None
        for member in zip_ref.namelist():
            if member.endswith(csv_file_name_in_zip):
                file_to_extract = member
                break
        
        if file_to_extract:
            zip_ref.extract(file_to_extract, path=data_dir)
            print(f"Extracted '{file_to_extract}' to '{data_dir}'.")
            extracted_file_path_actual = os.path.join(data_dir, file_to_extract)
            final_csv_path = os.path.join(data_dir, desired_csv_output_path)
            if extracted_file_path_actual != final_csv_path:
                os.makedirs(os.path.dirname(final_csv_path), exist_ok=True)
                os.rename(extracted_file_path_actual, final_csv_path)
                print(f"Moved '{extracted_file_path_actual}' to '{final_csv_path}'.")
            extracted_folder = os.path.dirname(file_to_extract)
            if extracted_folder and os.path.exists(os.path.join(data_dir, extracted_folder)) and not os.listdir(os.path.join(data_dir, extracted_folder)):
                os.rmdir(os.path.join(data_dir, extracted_folder))
                print(f"Cleaned up empty directory '{os.path.join(data_dir, extracted_folder)}'.")
            extracted_correctly = True
        else:
            print(f"Error: '{csv_file_name_in_zip}' not found within the zip file. Available files: {zip_ref.namelist()}")
    
    if os.path.exists(zip_file_name):
        os.remove(zip_file_name)
        print(f"Cleaned up '{zip_file_name}'.")
else:
    print(f"'{desired_csv_output_path}' already exists. Skipping download and extraction.")

# Load bank-full.csv into a pandas DataFrame
df_original = None # Keep original df for reference if needed
if os.path.exists(desired_csv_output_path):
    try:
        df_original = pd.read_csv(desired_csv_output_path, sep=';')
        df = df_original.copy() # Work with a copy for preprocessing
        print(f"\n'{desired_csv_output_path}' loaded successfully into DataFrame.")
    except Exception as e:
        print(f"Error loading '{desired_csv_output_path}': {e}")
        df = None
else:
    print(f"Skipping DataFrame load as '{desired_csv_output_path}' was not found.")
    df = None

## 3. Initial Exploratory Data Analysis (EDA)

In [ ]:
if df is not None:
    print("Dataset Shape:")
    print(df.shape)
else:
    print("DataFrame not loaded. Skipping EDA.")

In [ ]:
if df is not None:
    print("\nFirst 5 rows:")
    display(df.head())

In [ ]:
if df is not None:
    print("\nConcise summary of the DataFrame:")
    df.info()

In [ ]:
if df is not None:
    print("\nDescriptive statistics:")
    display(df.describe(include='all'))

In [ ]:
if df is not None:
    print("\nMissing values count (before handling 'unknown'):")
    print(df.isnull().sum())

## 4. Identify Numerical and Categorical Features

In [ ]:
if df is not None:
    numerical_features = df.select_dtypes(include=np.number).columns.tolist()
    categorical_features = df.select_dtypes(include='object').columns.tolist()
    
    if 'y' in categorical_features:
        target_column = 'y'
        categorical_features.remove(target_column) 
    elif 'y' in numerical_features: # If 'y' happens to be numerical already (e.g. 0/1)
        target_column = 'y'
        # We will still treat it as target, not a typical numerical feature for scaling initially
    else:
        target_column = None
        print("Warning: Target column 'y' not found.")
        
    print("Original Numerical Features:", numerical_features)
    print("Original Categorical Features (excluding target):", categorical_features)

## 5. Detailed EDA - Visualizations

### 5.1. Target Variable Distribution

In [ ]:
if df is not None and target_column and target_column in df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x=target_column, data=df)
    plt.title('Distribution of Target Variable (y)')
    plt.show()
    print(df[target_column].value_counts(normalize=True))

### 5.2. Numerical Features Analysis

In [ ]:
if df is not None and numerical_features:
    # Exclude target if it was initially numerical for these general plots
    plot_numerical_features = [f for f in numerical_features if f != target_column]
    if plot_numerical_features:
        print("\nHistograms for Numerical Features:")
        df[plot_numerical_features].hist(figsize=(12, 10), bins=20)
        plt.tight_layout()
        plt.show()

In [ ]:
if df is not None and numerical_features and target_column and target_column in df.columns:
    plot_numerical_features = [f for f in numerical_features if f != target_column]
    if plot_numerical_features:
        print("\nBoxplots for Numerical Features vs. Target Variable 'y':")
        for feature in plot_numerical_features:
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=target_column, y=feature, data=df)
            plt.title(f'{feature} vs. {target_column}')
            plt.show()

### 5.3. Categorical Features Analysis

In [ ]:
if df is not None and categorical_features:
    print("\nCount Plots for Categorical Features:")
    for feature in categorical_features:
        if df[feature].nunique() < 20: 
            plt.figure(figsize=(10, 6))
            sns.countplot(y=feature, data=df, order = df[feature].value_counts().index)
            plt.title(f'Distribution of {feature}')
            plt.tight_layout()
            plt.show()
        else:
            print(f"Skipping plot for {feature} as it has too many unique values ({df[feature].nunique()}).")

In [ ]:
if df is not None and categorical_features and target_column and target_column in df.columns:
    print("\nCount Plots for Categorical Features vs. Target Variable 'y':")
    for feature in categorical_features:
        if df[feature].nunique() < 20: 
            plt.figure(figsize=(12, 7))
            sns.countplot(x=feature, hue=target_column, data=df)
            plt.title(f'{feature} vs. {target_column}')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        else:
            print(f"Skipping plot for {feature} vs. {target_column} as {feature} has too many unique values ({df[feature].nunique()}).")

### 5.4. Correlation Analysis

In [ ]:
if df is not None and numerical_features:
    # Temporarily convert target to numeric for correlation if it's not already
    df_corr = df.copy()
    if target_column and df_corr[target_column].dtype == 'object':
      df_corr[target_column] = df_corr[target_column].map({'yes': 1, 'no': 0}).fillna(-1) # temp encoding for corr
    
    # Include target in numerical features for correlation matrix if it's binary encoded
    corr_features = [f for f in numerical_features if f != target_column] 
    if target_column and df_corr[target_column].dtype != 'object':
        corr_features.append(target_column)
        
    if corr_features:
        print("\nCorrelation Matrix (Numerical Features and Encoded Target):")
        corr_matrix = df_corr[corr_features].corr()
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
        plt.title('Correlation Matrix')
        plt.show()
        display(corr_matrix)

## 6. Handle 'unknown' values

In [ ]:
if df is not None:
    print("Columns with 'unknown' values before handling:")
    cols_with_unknown = []
    for col in df.columns:
        # Check if 'unknown' is in unique values, handling potential float types
        if df[col].dtype == 'object' and 'unknown' in df[col].unique():
            cols_with_unknown.append(col)
            print(f"Column '{col}': {df[col].value_counts()['unknown']} 'unknown' values")
    
    if not cols_with_unknown:
        print("No columns with 'unknown' values found.")
    else:
        # Replace 'unknown' with np.nan
        df.replace('unknown', np.nan, inplace=True)
        print("\nReplaced 'unknown' with np.nan.")
    
    # Impute NaN values with the mode for each column
    print("\nImputing NaN values with mode...")
    for column in df.columns:
        if df[column].isnull().any():
            mode_val = df[column].mode()[0]
            df[column].fillna(mode_val, inplace=True)
            print(f"Filled NaNs in '{column}' with mode: {mode_val}")
            
    print("\nMissing values count after handling 'unknown' and imputation:")
    print(df.isnull().sum().sum(), "total missing values.")

## 7. Outlier Detection and Handling

The boxplots for numerical features (e.g., 'age', 'balance', 'duration', 'campaign', 'pdays', 'previous') indicate the presence of outliers. 
- 'age': Some older individuals.
- 'balance': Significant number of high positive and some negative balances (overdrafts).
- 'duration': Some very long call durations. Note: This feature is highly correlated with the target and should be handled carefully (e.g., not used as input if it's post-call information, or if it's used, its nature must be understood).
- 'campaign': Some customers contacted many times.
- 'pdays': Many -1 values (not previously contacted), and then a spread for those contacted.
- 'previous': Some customers contacted multiple times before.

**Strategy for this iteration:**
For this initial pipeline, we will note these outliers but not implement aggressive outlier removal or transformation. This is to preserve as much original data as possible and establish a baseline. 

**Potential future strategies if outliers prove problematic:**
1.  **Capping/Winsorization**: Limiting extreme values to a certain percentile (e.g., 1st and 99th).
2.  **Transformation**: Applying log, square root, or Box-Cox transformations to reduce skewness and outlier impact.
3.  **Removal**: If outliers are deemed to be errors or are extremely influential and not representative, they could be removed, but this should be done cautiously.
4.  **Using robust models**: Some models are inherently more robust to outliers.

## 8. Data Preprocessing - Encoding

### 8.1. Binary Categorical Features

In [ ]:
if df is not None and target_column:
    binary_cols = ['default', 'housing', 'loan'] # Target 'y' will be handled separately if not already 0/1
    if target_column in df.columns and df[target_column].dtype == 'object':
        binary_cols.append(target_column)
        
    for col in binary_cols:
        if col in df.columns and df[col].dtype == 'object': # Check if it's object type before mapping
            # Make sure only 'yes' and 'no' are present, or handle other cases
            unique_vals = df[col].unique()
            if set(unique_vals) <= {'yes', 'no'}:
                 df[col] = df[col].map({'yes': 1, 'no': 0})
                 print(f"Encoded '{col}': {df[col].value_counts(dropna=False).to_dict()}")
            else:
                 print(f"Warning: Column '{col}' is not strictly 'yes'/'no' and was not binary encoded. Values: {unique_vals}")
        elif col in df.columns and df[col].dtype != 'object':
             print(f"Column '{col}' is already numeric. Skipping binary encoding.")
        else:
            print(f"Warning: Binary column '{col}' not found in DataFrame for encoding.")
    
    # Ensure target 'y' is 0/1 and update its status if encoding happened
    if target_column and target_column in df.columns and df[target_column].dtype != 'object':
      if target_column not in numerical_features: numerical_features.append(target_column)
      if target_column in categorical_features: categorical_features.remove(target_column)

### 8.2. Other Categorical Features (One-Hot Encoding)

In [ ]:
if df is not None:
    # Re-identify categorical features after potential 'unknown' handling and binary encoding
    categorical_features_to_encode = df.select_dtypes(include='object').columns.tolist()
    # Ensure target is not in this list if it was object and now encoded, or was already numeric
    if target_column in categorical_features_to_encode:
        categorical_features_to_encode.remove(target_column)
        
    print(f"\nCategorical features to one-hot encode: {categorical_features_to_encode}")
    
    if categorical_features_to_encode:
        df_encoded = pd.get_dummies(df, columns=categorical_features_to_encode, drop_first=True, dummy_na=False) # dummy_na=False is default
        print("\nDataFrame shape before one-hot encoding:", df.shape)
        print("DataFrame shape after one-hot encoding:", df_encoded.shape)
        # display(df_encoded.head())
        df = df_encoded # Update df to the new encoded dataframe
    else:
        print("\nNo remaining categorical features to one-hot encode.")
    
    # Display final columns to check for target
    # print("\nFinal DataFrame columns:", df.columns.tolist())

## 9. Data Preprocessing - Scaling Numerical Features

In [ ]:
if df is not None and target_column and target_column in df.columns:
    current_numerical_features = df.select_dtypes(include=np.number).columns.tolist()
    
    features_to_scale = [col for col in current_numerical_features if col != target_column]

    print(f"\nNumerical features to scale (excluding target '{target_column}'): {features_to_scale}")
    
    if features_to_scale:
        scaler = StandardScaler()
        df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
        print("\nNumerical features scaled using StandardScaler.")
        # print("\nFirst 5 rows after scaling:")
        # display(df.head())
    else:
        print("\nNo numerical features to scale.")
else:
    print("DataFrame or target column not available for scaling.")

## 10. Save Processed Data

In [ ]:
processed_file_path = None # Initialize
if df is not None:
    processed_file_path = 'bank-full-processed.csv'
    df.to_csv(processed_file_path, index=False)
    print(f"\nProcessed DataFrame saved to '{processed_file_path}'")
    
    # Verify by loading it back (optional)
    # df_loaded_processed = pd.read_csv(processed_file_path)
    # display(df_loaded_processed.head())
    # print(f"Shape of loaded processed data: {df_loaded_processed.shape}")
else:
    print("DataFrame not available. Skipping saving of processed data.")

## 11. Feature Selection using Mutual Information Gain

Now that the data is preprocessed (handled 'unknowns', encoded, scaled), we will apply feature selection to identify the most relevant features for predicting the target variable 'y'. We will use Mutual Information Gain for this purpose.

In [ ]:
df_for_selection = None
if processed_file_path and os.path.exists(processed_file_path):
    try:
        df_for_selection = pd.read_csv(processed_file_path)
        print(f"Loaded '{processed_file_path}' for feature selection. Shape: {df_for_selection.shape}")
    except Exception as e:
        print(f"Error loading '{processed_file_path}': {e}")
        df_for_selection = None
elif df is not None: # Fallback to use the df in memory if file saving/loading failed
    print("Using DataFrame from memory for feature selection.")
    df_for_selection = df.copy()
else:
    print("Preprocessed data not available for feature selection.")

X = None
y_fs = None # y for feature selection

if df_for_selection is not None and target_column and target_column in df_for_selection.columns:
    # Ensure target 'y' is integer type for mutual_info_classif
    if df_for_selection[target_column].dtype == 'float': # Can happen if it was binary (0.0/1.0)
        df_for_selection[target_column] = df_for_selection[target_column].astype(int)
        print(f"Converted target column '{target_column}' to integer for feature selection.")
        
    if df_for_selection[target_column].isnull().any():
        print(f"Warning: Target column '{target_column}' contains NaN values. This might affect feature selection.")
        # Optionally, handle NaNs here, e.g., by dropping rows with NaN target
        # df_for_selection.dropna(subset=[target_column], inplace=True)
        # print(f"Dropped rows with NaN target. New shape: {df_for_selection.shape}")
        
    X = df_for_selection.drop(columns=[target_column])
    y_fs = df_for_selection[target_column]
    print(f"Prepared X (features shape: {X.shape}) and y (target shape: {y_fs.shape}) for feature selection.")
    
    # Ensure all features in X are numeric (they should be after preprocessing)
    non_numeric_cols = X.select_dtypes(exclude=np.number).columns.tolist()
    if non_numeric_cols:
        print(f"Warning: Non-numeric columns found in X: {non_numeric_cols}. This will cause an error in mutual_info_classif.")
        # Attempt to coerce, or drop, or raise error
        # For now, let's assume they should have been handled and proceed, an error will occur if not.
else:
    print("Feature selection cannot proceed as data or target column is not properly set up.")

In [ ]:
mi_scores = None
if X is not None and y_fs is not None:
    try:
        # Calculate mutual information scores
        # Ensure y_fs does not contain NaNs if not handled before
        if y_fs.isnull().any():
            print("Target variable y_fs contains NaNs. Cleaning up before mutual_info_classif...")
            X_temp = X[~y_fs.isnull()]
            y_fs_temp = y_fs[~y_fs.isnull()]
            if y_fs_temp.empty:
                raise ValueError("Target variable y_fs is all NaNs after cleanup.")
            print(f"Shapes after NaN target removal for MI calc: X_temp={X_temp.shape}, y_fs_temp={y_fs_temp.shape}")
            mi_scores = mutual_info_classif(X_temp, y_fs_temp, random_state=42)
        else:
            mi_scores = mutual_info_classif(X, y_fs, random_state=42)
        
        mi_scores_series = pd.Series(mi_scores, name='MI_Score', index=X.columns)
        mi_scores_series = mi_scores_series.sort_values(ascending=False)
        
        print("\nMutual Information Scores (Top 30):")
        display(mi_scores_series.head(30))
        
        # Plotting the MI scores
        plt.figure(figsize=(12, 8))
        mi_scores_series.head(30).plot(kind='bar')
        plt.title('Top 30 Features by Mutual Information Score')
        plt.ylabel('Mutual Information Score')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error during mutual information calculation or plotting: {e}")
        # print traceback for more details if running interactively
        # import traceback
        # traceback.print_exc()
else:
    print("X or y_fs not available for mutual information calculation.")

In [ ]:
selected_features_df = None
selected_features_list = []
K = 20 # Desired number of top features

if mi_scores_series is not None and X is not None and y_fs is not None:
    num_total_features = X.shape[1]
    k_to_select = min(K, num_total_features) # Select K or all if less than K
    
    print(f"\nSelecting top {k_to_select} features based on MI scores...")
    
    # Using SelectKBest with mutual_info_classif
    # Need to handle potential NaNs in y_fs for SelectKBest as well
    if y_fs.isnull().any():
        print("Target variable y_fs contains NaNs. Cleaning up before SelectKBest...")
        X_select = X[~y_fs.isnull()].copy() # Use .copy() to avoid SettingWithCopyWarning
        y_select = y_fs[~y_fs.isnull()].copy()
        if y_select.empty:
            print("Error: Target variable y_fs is all NaNs after cleanup. Cannot proceed with SelectKBest.")
            selector = None
        else:
            print(f"Shapes after NaN target removal for SelectKBest: X_select={X_select.shape}, y_select={y_select.shape}")
            selector = SelectKBest(score_func=mutual_info_classif, k=k_to_select)
            selector.fit(X_select, y_select)
            selected_features_mask = selector.get_support()
            selected_features_list = X_select.columns[selected_features_mask].tolist()
    else:
        selector = SelectKBest(score_func=mutual_info_classif, k=k_to_select)
        selector.fit(X, y_fs)
        selected_features_mask = selector.get_support()
        selected_features_list = X.columns[selected_features_mask].tolist()
        
    if selected_features_list:
        print(f"\nSelected {len(selected_features_list)} features:")
        print(selected_features_list)
        
        # Create a new DataFrame with selected features and the target
        # Use the original X and y_fs to ensure all rows are included, then drop NaNs if y_fs had them initially
        selected_features_df = X[selected_features_list].copy() # .copy() to avoid SettingWithCopyWarning
        selected_features_df[target_column] = y_fs.values # Add target variable back
        
        # If y_fs had NaNs, those rows will have NaN target in selected_features_df
        # Depending on strategy, might drop them now or let training pipeline handle
        # For now, keep them to match original row count, training pipeline should be aware
        # if y_fs.isnull().any():
        #    print(f"Target column had NaNs. Resulting selected_features_df might have NaNs in target. Shape: {selected_features_df.shape}")
            
        print("\nFirst 5 rows of DataFrame with selected features and target:")
        display(selected_features_df.head())
    else:
        print("No features were selected. This might indicate an issue.")
else:
    print("MI scores or data not available. Skipping feature selection based on KBest.")

## 12. Save Final Selected Features Data and List

In [ ]:
selected_features_file_path = None
selected_features_list_path = None

if selected_features_df is not None:
    selected_features_file_path = 'bank-features-selected.csv'
    selected_features_df.to_csv(selected_features_file_path, index=False)
    print(f"\nDataFrame with selected features saved to '{selected_features_file_path}'")
    print(f"Shape of saved selected features data: {selected_features_df.shape}")
    
    # Save the list of selected feature names as well (e.g., in a JSON file)
    if selected_features_list:
        selected_features_list_path = 'selected_feature_names.json'
        with open(selected_features_list_path, 'w') as f:
            json.dump(selected_features_list, f)
        print(f"List of selected feature names saved to '{selected_features_list_path}'")
else:
    print("DataFrame with selected features is not available. Skipping saving.")

### Feature Selection Summary:
We employed Mutual Information Gain to assess the relevance of each feature with respect to the target variable 'y'. Mutual information measures the amount of information obtained about one random variable through observing the other. Higher values indicate a stronger relationship.

Based on these scores, we selected the top K features. In this iteration, K was set to 20 (or fewer if the total number of features was less than 20). The resulting DataFrame, containing only these selected features and the target variable, has been saved to `bank-features-selected.csv`. A JSON file `selected_feature_names.json` containing the list of these feature names has also been saved. This dataset is now prepared for the model training phase.

## 13. Observations & Summary of Preprocessing and Feature Selection

### EDA Findings:
- **Target Variable ('y')**: The dataset is imbalanced (approx. 88% 'no', 12% 'yes').
- **Numerical Features**: Varied distributions, outliers present (e.g., 'balance', 'duration'). 'duration' is highly correlated with the target but might be a leakage variable.
- **Categorical Features**: 'unknown' values were present and handled. Distributions varied. Relationships with 'y' observed.

### Preprocessing Steps Undertaken:
1.  **Loaded Data**: `bank-full.csv`.
2.  **Handled 'unknown' Values**: Replaced with `np.nan`, then imputed with mode.
3.  **Encoded Binary Features**: 'default', 'housing', 'loan', 'y' to 1/0.
4.  **One-Hot Encoded Categorical Features**: Using `pd.get_dummies(drop_first=True)`.
5.  **Scaled Numerical Features**: Using `StandardScaler` (excluding target 'y').
6.  **Saved Processed Data**: To `bank-full-processed.csv`.

### Feature Selection Steps:
1.  **Calculated Mutual Information Scores**: Between each feature and the target 'y' using the preprocessed data.
2.  **Selected Top K Features**: Aimed for K=20 features with the highest MI scores. If total features < 20, all were selected. The actual number of selected features is reported in the output above.
3.  **Saved Selected Features Data**: The DataFrame with selected features and target saved to `bank-features-selected.csv`.
4.  **Saved Selected Feature Names**: The list of selected feature names saved to `selected_feature_names.json`.

### Next Steps:
The dataset `bank-features-selected.csv` containing the most relevant features is now ready for the `model_training.ipynb` notebook.